# Eksperimenter med lånekassens dokumenter med NB sBERT 
De har mye parallelldata

In [1]:
from pathlib import Path
import pandas as pd
from hemmelig import data_path

p = Path(data_path)

filer = [e for e in p.iterdir() if "lanekassen" in e.name]

dfs = []
for e in filer:
    with e.open("rb") as f:
        dfs.append(pd.read_json(e, lines=True))


df = pd.concat(dfs)
df

,doc_hash,lang,url,domain,date,mimetype,fulltext
0,b5fc6f81d5709f0051da1964c3900e7ca8df65f7,nno,http://lanekassen.no/globalassets/brosjyrer-fe...,lanekassen.no,2022-12-19 01:53:28+00:00,pdf,[Er du flyktning? Du blir rekna som flyktning ...
1,1fbe096fdfdd9916be1313006ac5cf44ac650231,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:14+00:00,pdf,"[, , Du kan bruke dette skjemaet dersom du tar..."
2,582aaf9a510b3bde21b5b3e13ffc3453d6ca8174,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:21+00:00,pdf,"[, , Det er viktig at du les informasjonen på ..."
3,32e23cf05e4db850a9e2f622c2edd0482572677e,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:00:45+00:00,pdf,[Nynorsk Skjema for lærlinglønn Kor stort bort...
4,4ea46648c7ce59fe0c39b45779ed6402d8b41369,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:01:01+00:00,pdf,[Nynorsk Skjema I – artikkelnr. 9001550 – nyno...
...,...,...,...,...,...,...,...
72,e29a43da7165d92d937c56416d50f563201ab5fa,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:18+00:00,pdf,"[, , 01.01.2020 Storebrand Bank ASA 70 % Bolig..."
73,c120ee7f582c758ae392b8f90795a7c664c39e90,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:23+00:00,pdf,"[, , 06.11.2019 Storebrand Bank ASA 70 % Bolig..."
74,553101a9b0a9fac935cf31e5dd59555d23ecbefe,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:41+00:00,pdf,"[, , Flyktningstipendet Mottakere av flyktning..."
75,1f14c7bcc564613f79be7a8cae1a946e27cfd755,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:44+00:00,pdf,"[, , Tall og fakta om Lånekassens kunder og or..."


In [2]:
nynorske = df[df.lang == "nno"]
nynorske.index = range(len(nynorske))

bokmålske = df[df.lang == "nob"]
bokmålske.index = range(len(bokmålske))

len(nynorske), len(bokmålske)

(256, 310)

Lim sammen avsnittene i hvert dokument

In [3]:
nynorsk_texts_joined = nynorske.fulltext.apply(lambda x: "\n".join(x))
bokmål_texts_joined = bokmålske.fulltext.apply(lambda x: "\n".join(x))

Last inn fasit

In [4]:
hash_to_i_nn = {e.doc_hash: e.Index for e in nynorske.itertuples()}
hash_to_i_bm = {e.doc_hash: e.Index for e in bokmålske.itertuples()}

fasit = pd.read_csv("lanekassen_fasit.csv")
fasit_set = set(zip(fasit.nynorsk_doc_hash, fasit.bokmål_doc_hash))

def compare_matches_to_fasit(matches):
    nn_i, bm_i = zip(*[(i, match["corpus_id"]) for i, match in matches])
    hash_matches = set(zip(nynorske.iloc[list(nn_i)].doc_hash, bokmålske.iloc[list(bm_i)].doc_hash))

    hits = fasit_set.intersection(hash_matches)
    misses = fasit_set - hash_matches
    
    hits_indices = [(hash_to_i_nn[nn_doc_hash], hash_to_i_bm[bm_doc_hash]) for nn_doc_hash, bm_doc_hash in hits]
    misses_indices = [(hash_to_i_nn[nn_doc_hash], hash_to_i_bm[bm_doc_hash]) for nn_doc_hash, bm_doc_hash in misses]
    
    return (hits_indices, misses_indices)

Last inn modellen

In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('NbAiLab/nb-sbert-base', device="cuda")

# Tell hvor mange dokumenter og avsnitt som er for lange for modellen

In [6]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

nynorsk_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in nynorsk_texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in nynorsk_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(nynorsk_documents_token_sequence_lengths)} nynorske dokumentene er:
    {under_max_len} under NB sBERT sin makslengde 
    {over_max_len}  over NB sBERT sin makslengde 
Altså er {round(under_max_len/len(nynorsk_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")

bokmål_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in bokmål_texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in bokmål_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(bokmål_documents_token_sequence_lengths)} dokumentene på bokmål er:
    {under_max_len} under NB sBERT sin makslengde 
    {over_max_len}  over NB sBERT sin makslengde 
Altså er {round(under_max_len/len(bokmål_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")


Av de 256 nynorske dokumentene er:
    2 under NB sBERT sin makslengde 
    254  over NB sBERT sin makslengde 
Altså er 0.78% av dokumentene under makslengden 


Av de 310 dokumentene på bokmål er:
    1 under NB sBERT sin makslengde 
    309  over NB sBERT sin makslengde 
Altså er 0.32% av dokumentene under makslengden 



In [7]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

token_sequence_lengths = df.fulltext.apply(lambda x: [len(tokenizer.tokenize(e)) for e in x])

under_max_len = 0
over_max_len = 0
zero_len = 0
for e in token_sequence_lengths:
    for token_len in e:
        if token_len:
            if token_len > max_len:
                over_max_len += 1
            else:
                under_max_len += 1
        else:
            zero_len += 1

print(f"""
Det er totalt {sum((under_max_len, over_max_len))} ikke-tomme avsnitt/setninger (og {zero_len} er tomme)
Av de ikke-tomme avsnittene er:
    {under_max_len} under NB sBERT sin makslengde 
    {over_max_len}  over NB sBERT sin makslengde 
Altså er {round(under_max_len/sum((under_max_len, over_max_len))*100, 2)}% av de ikke-tomme avsnittene under makslengden 
""")



Det er totalt 12491 ikke-tomme avsnitt/setninger (og 4377 er tomme)
Av de ikke-tomme avsnittene er:
    8877 under NB sBERT sin makslengde 
    3614  over NB sBERT sin makslengde 
Altså er 71.07% av de ikke-tomme avsnittene under makslengden 



# Dokumentalignment 

In [8]:
doc_alignment_results = {}

## Aksepter cut-off
Lim sammen avsnittene i hvert dokument, kjør gjennom sbert

In [9]:
import numpy as np

emb_path = Path("embeddings/nb_sbert/texts_joined.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    bokmål_embeddings = model.encode(bokmål_texts_joined)
    nynorsk_embeddings = model.encode(nynorsk_texts_joined)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["naiv_cutoff"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

### Inspiser matches

In [10]:
from utils import print_matches

print_matches(matches, nynorsk_texts_joined, bokmål_texts_joined)
print_matches(non_matches, nynorsk_texts_joined, bokmål_texts_joined)



Søketekst: 
        Er du flyktning? Du blir rekna som flyktning dersom du er utanlandsk statsborgar og har fått vern (asyl) i Noreg. Du må vere registrert med flyktningstatus hos Utlendingsdirektoratet (UDI). Du kan få stipend og lån Er du flyktning, kan du få stipend og lån til all utdanning i Noreg som er godkjend f
Match:
        Er du flyktning? Du regnes som flyktning hvis du er utenlandsk statsborger og har fått beskyttelse (asyl) i Norge. Du må være registrert med flyktningstatus hos Utlendingsdirektoratet (UDI). Du kan få stipend og lån Er du flyktning, kan du få stipend og lån til all utdanning i Norge som er godkjent 
Likhet:
        0.9900512099266052
        
Indekser:
            Nynorsk/søketekst:  0
            Bokmål/treff:       241
_______________________________________________________________

Søketekst: 
        

Det er viktig at du les informasjonen på side 2 i dette søknadsskjemaet. Hugs å sende inn ein , dersom du ikkje har gjort det.

Denne søknaden lastar d

#### Falsk positiv
Eksempel på en falsk positiv.  
Noen lister av datoer og banker og renter blir veldig like (similarity score er ~0.98)

In [11]:
i = 19 
print(search_result[i])
print(nynorsk_texts_joined[i][:250])
print("\n___________\n")
print(bokmål_texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 291, 'score': 0.9755687117576599}]


04.05.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,79% 04.05.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,97% 04.05.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 2,07% 04.05.2022 Sunndal Sparebank B

___________



02.03.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1.79% 02.03.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1.81% 02.03.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1.97% 02.03.2022 KLP Banken AS Bolig


## Del opp avsnittene i biter mindre enn modellens makslengde og aggreger

In [12]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in nynorsk_texts_joined]
bokmål_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in bokmål_texts_joined]

In [13]:
# from collections import Counter 
# pd.DataFrame(Counter([len(e) for e in nynorsk_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")
# pd.DataFrame(Counter([len(e) for e in bokmål_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")

## Bare send alle bitene gjennom

SentenceTransformers aggregerer ved å ta de to første tekstene i hver liste av tekster, og lage en vektor hvor hver halvdel er starten av hver av tekstene (fra starten og opp til 1/2 makslengde) adskilt med et sep-token

In [14]:
emb_path = Path("embeddings/nb_sbert/maxlen_parts.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    nynorsk_maxlen_parts_1 = [e + [""] for e in nynorsk_maxlen_parts]
    bokmål_maxlen_parts_1 = [e + [""] for e in bokmål_maxlen_parts]

    bokmål_embeddings = model.encode(bokmål_maxlen_parts_1)
    nynorsk_embeddings = model.encode(nynorsk_maxlen_parts_1)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["naiv_cutoff_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

### Inspiser matches

In [15]:
print_matches(matches, nynorsk_texts_joined, bokmål_texts_joined)
# print_matches(non_matches, nynorsk_texts_joined, bokmål_texts_joined)



Søketekst: 
        Er du flyktning? Du blir rekna som flyktning dersom du er utanlandsk statsborgar og har fått vern (asyl) i Noreg. Du må vere registrert med flyktningstatus hos Utlendingsdirektoratet (UDI). Du kan få stipend og lån Er du flyktning, kan du få stipend og lån til all utdanning i Noreg som er godkjend f
Match:
        Er du flyktning? Du regnes som flyktning hvis du er utenlandsk statsborger og har fått beskyttelse (asyl) i Norge. Du må være registrert med flyktningstatus hos Utlendingsdirektoratet (UDI). Du kan få stipend og lån Er du flyktning, kan du få stipend og lån til all utdanning i Norge som er godkjent 
Likhet:
        0.9803029894828796
        
Indekser:
            Nynorsk/søketekst:  0
            Bokmål/treff:       241
_______________________________________________________________

Søketekst: 
        

Det er viktig at du les informasjonen på side 2 i dette søknadsskjemaet. Hugs å sende inn ein , dersom du ikkje har gjort det.

Denne søknaden lastar d

#### Falsk positiv
Samme som over

In [16]:
i = 22
print(search_result[i])
print(nynorsk_texts_joined[i][:250])
print("\n___________\n")
print(bokmål_texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 293, 'score': 0.9541565179824829}]


05.05.2021 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,29 % 05.05.2021 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1,39 % 05.05.2021 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,57 % 05.05.2021 Sunndal Spareban

___________



03.11.2021 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,29 % 03.11.2021 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,47 % 03.11.2021 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1,56 % 03.11.2021 Sunndal Spareban


## Aggreger setningsvektorene: Mean pooling

In [17]:
import numpy as np 

emb_path = Path("embeddings/nb_sbert/maxlen_parts_mean_pooling.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    nynorsk_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

hits, misses = compare_matches_to_fasit(matches)

doc_alignment_results["mean_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

### Inspiser matches

In [18]:
print_matches(matches, nynorsk_texts_joined, bokmål_texts_joined)


Søketekst: 
        Er du flyktning? Du blir rekna som flyktning dersom du er utanlandsk statsborgar og har fått vern (asyl) i Noreg. Du må vere registrert med flyktningstatus hos Utlendingsdirektoratet (UDI). Du kan få stipend og lån Er du flyktning, kan du få stipend og lån til all utdanning i Noreg som er godkjend f
Match:
        Er du flyktning? Du regnes som flyktning hvis du er utenlandsk statsborger og har fått beskyttelse (asyl) i Norge. Du må være registrert med flyktningstatus hos Utlendingsdirektoratet (UDI). Du kan få stipend og lån Er du flyktning, kan du få stipend og lån til all utdanning i Norge som er godkjent 
Likhet:
        0.9855220317840576
        
Indekser:
            Nynorsk/søketekst:  0
            Bokmål/treff:       241
_______________________________________________________________

Søketekst: 
        

Du kan bruke dette skjemaet dersom du tar høgare utdanning eller fagskoleutdanning i Noreg eller Norden, og du skal søke om støtte til studieopphald i 

In [19]:
for nn_i, bm_i in misses:
    match_i = matches[nn_i][1]["corpus_id"]
    print(f"Nynorsktekst indeks: {nn_i} \nFasit indeks: {bm_i} \nSøketreff indeks: {match_i}\nSøketreff score: {matches[nn_i][1]['score']}")
    print("_____\nNynorsk tekst:")
    print(nynorsk_texts_joined[nn_i][:300])
    print("\n_____\nFasit tekst:")
    print(bokmål_texts_joined[bm_i][:300])
    print("\n_____\nSøketreff tekst:")
    print(bokmål_texts_joined[match_i][:300])
    print("\n--------------------------\n")


Nynorsktekst indeks: 153 
Fasit indeks: 122 
Søketreff indeks: 133
Søketreff score: 0.9754108786582947
_____
Nynorsk tekst:
Ofte stilte spørsmål
Alle lærestader med utdanningar som er godkjente for utdanningsstøtte i Lånekassen, har ein administrator i arbeidsflata. Administratoren er ansvarleg for kven som har tilgang til arbeidsflata for lærestaden. Viss du manglar tilgang, kan administratoren ved din lærestad hjelpe d

_____
Fasit tekst:
Ofte stilte spørsmål
Alle læresteder med utdanninger som er godkjent for utdanningsstøtte i Lånekassen, har en administrator i arbeidsflaten. Administratoren er ansvarlig for hvem som har tilgang til arbeidsflaten for lærestedet. Dersom du mangler tilgang, kan administratoren ved ditt lærested hjelp

_____
Søketreff tekst:
– Fastrenten har aldri vært så mye høyere enn flytende rente som den er fra 1. november i år, sier administrerende direktør i Lånekassen, Morten Rosenkvist.
– Samtidig blir renten for å binde i tre år høyere enn for både 5 og 10 å

#### Falsk positiv:

In [20]:
i = 19
print(search_result[i])
print(nynorsk_texts_joined[i][:250])
print("\n___________\n")
print(bokmål_texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 291, 'score': 0.9914844036102295}]


04.05.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,79% 04.05.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,97% 04.05.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 2,07% 04.05.2022 Sunndal Sparebank B

___________



02.03.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1.79% 02.03.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1.81% 02.03.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1.97% 02.03.2022 KLP Banken AS Bolig


## Aggreger setningsvektorene: Max pooling

In [21]:
import numpy as np 

emb_path = Path("embeddings/nb_sbert/maxlen_parts_max_pooling.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    nynorsk_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["max_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

### Inspiser matches

In [22]:
from utils import print_matches

print_matches(matches, nynorsk_texts_joined, bokmål_texts_joined)


Søketekst: 
        Er du flyktning? Du blir rekna som flyktning dersom du er utanlandsk statsborgar og har fått vern (asyl) i Noreg. Du må vere registrert med flyktningstatus hos Utlendingsdirektoratet (UDI). Du kan få stipend og lån Er du flyktning, kan du få stipend og lån til all utdanning i Noreg som er godkjend f
Match:
        Er du flyktning? Du regnes som flyktning hvis du er utenlandsk statsborger og har fått beskyttelse (asyl) i Norge. Du må være registrert med flyktningstatus hos Utlendingsdirektoratet (UDI). Du kan få stipend og lån Er du flyktning, kan du få stipend og lån til all utdanning i Norge som er godkjent 
Likhet:
        0.9942201972007751
        
Indekser:
            Nynorsk/søketekst:  0
            Bokmål/treff:       241
_______________________________________________________________

Søketekst: 
        

Du kan bruke dette skjemaet dersom du tar høgare utdanning eller fagskoleutdanning i Noreg eller Norden, og du skal søke om støtte til studieopphald i 

#### Falsk positiv:
Den samme som over

In [23]:
i = 19
print(search_result[i])
print(nynorsk_texts_joined[i][:250])
print("\n___________\n")
print(bokmål_texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 293, 'score': 0.961942195892334}]


04.05.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,79% 04.05.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,97% 04.05.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 2,07% 04.05.2022 Sunndal Sparebank B

___________



03.11.2021 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,29 % 03.11.2021 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,47 % 03.11.2021 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1,56 % 03.11.2021 Sunndal Spareban


## Konklusjon

In [24]:
pd.DataFrame(doc_alignment_results).T.sort_values("hits")

,matches,threshold,percent of docs,hits,misses
naiv_cutoff_biter,180.0,0.95,70.31,40.0,15.0
naiv_cutoff,204.0,0.95,79.69,42.0,13.0
max_pooling_biter,221.0,0.95,86.33,48.0,7.0
mean_pooling_biter,236.0,0.95,92.19,52.0,3.0


Vi får flest matches og flest hits ved å aggregere opp sentence embeddings med mean pooling.   
Vi har ikke så mange negative eksempler, så det er ikke helt sikkert at dette er den mest nøyaktige metoden.

# Setnings/avsnittsalignment

In [25]:
sent_alignment_results = {}

In [26]:
nynorsk_texts_flat = sorted(list({text for text_list in nynorske.fulltext for text in text_list if text}))
bokmål_texts_flat = sorted(list({text for text_list in bokmålske.fulltext for text in text_list if text}))
len(nynorsk_texts_flat), len(bokmål_texts_flat)

(4032, 6452)

In [27]:
nn_pair_indices = set()
bm_pair_indices = set()
for i, text in enumerate(nynorsk_texts_flat):
    for i2, text2 in enumerate(bokmål_texts_flat):
        if text == text2:
            nn_pair_indices.add(i)
            bm_pair_indices.add(i2)
len(nn_pair_indices)

sent_alignment_results["string_comparison"] = {"matches": len(nn_pair_indices), "percent of sents": round(len(nn_pair_indices)/len(nynorsk_texts_flat), 2)}

In [28]:
nynorsk_texts_flat = [text for i, text in enumerate(nynorsk_texts_flat) if i not in nn_pair_indices]
bokmål_texts_flat = [text for i, text in enumerate(bokmål_texts_flat) if i not in bm_pair_indices]

## Naiv approach
Godta cut-off på sBERT sin maxlengde

In [29]:
import numpy as np

emb_path = Path("embeddings/nb_sbert/texts_flat.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    nynorsk_embeddings = model.encode(nynorsk_texts_flat)
    bokmål_embeddings = model.encode(bokmål_texts_flat)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
sent_alignment_results["naiv_cutoff"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_texts_flat)*100, 2)}


### Inspiser matches

In [30]:
print_matches(matches, nynorsk_texts_flat, bokmål_texts_flat, stop_printing_at=30)


Søketekst: 
        ,må du først ha sendt inn ein søknad om lån og stipendfor det studieåret du får barn. , men har fått barn innan sju månader etter at du har fullført ein grad,sender du inn denne søknaden saman med dokumentasjon på graden du har fullført. Du sender inn denne søknaden og dokumentasjonen etter at du ha
Match:
        ,må du først ha sendt inn en søknad om lån og stipendfor det studieåret du får barn. , men har fått barn innen sju måneder etter at du har fullført en grad,sender du inn denne søknaden sammen med dokumentasjon på graden du har fullført. Du sender inn søknaden og dokumentasjonen etter at du har fått 
Likhet:
        0.9970904588699341
        
Indekser:
            Nynorsk/søketekst:  2
            Bokmål/treff:       9
_______________________________________________________________

Søketekst: 
        - Eg er stolt av arbeidsplassen min!
Match:
        - Jeg er stolt av arbeidsplassen min!
Likhet:
        0.9705960750579834
        
Indekser:
           

## Del opp setninger/avsnitt som er lengre enn sBERT sin maxlengde

In [31]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in nynorsk_texts_flat]
bokmål_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in bokmål_texts_flat]

## Naiv 2
Aggreger med Sententence_transformers hack (to første listene kuttes på 1/2 maxlen og konkateneres)

In [32]:
emb_path = Path("embeddings/nb_sbert/maxlen_parts_flat.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    nynorsk_embeddings = model.encode([e+[""] for e in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = model.encode([e+[""] for e in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
sent_alignment_results["naiv_cutoff_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

### Inspiser matches

In [33]:
print_matches(matches, nynorsk_texts_flat, bokmål_texts_flat, stop_printing_at=50)


Søketekst: 
        ,må du først ha sendt inn ein søknad om lån og stipendfor det studieåret du får barn. , men har fått barn innan sju månader etter at du har fullført ein grad,sender du inn denne søknaden saman med dokumentasjon på graden du har fullført. Du sender inn denne søknaden og dokumentasjonen etter at du ha
Match:
        ,må du først ha sendt inn en søknad om lån og stipendfor det studieåret du får barn. , men har fått barn innen sju måneder etter at du har fullført en grad,sender du inn denne søknaden sammen med dokumentasjon på graden du har fullført. Du sender inn søknaden og dokumentasjonen etter at du har fått 
Likhet:
        0.9964262843132019
        
Indekser:
            Nynorsk/søketekst:  2
            Bokmål/treff:       9
_______________________________________________________________

Søketekst: 
        - Eg er stolt av arbeidsplassen min!
Match:
        - Jeg er stolt av arbeidsplassen min!
Likhet:
        0.971623420715332
        
Indekser:
            

#### Falsk positiv

In [34]:
i = 24
print(search_result[i])
print(nynorsk_texts_flat[i])
print("\n___________\n")
print(bokmål_texts_flat[search_result[i][0]["corpus_id"]])

[{'corpus_id': 93, 'score': 0.9654372930526733}]
05.02.2020 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 2,68 % 05.02.2020 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 2,73 % 05.02.2020 Sbanken ASA Boliglån 50 % 2,75 % 05.02.2020 Fana Sparebank Nettlån Bolig 50 % 2,80 % 05.02.2020 Gjensidige Bank Boliglån inntil 50 % 2,91 %

___________

06.05.2020 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,92 % 06.05.2020 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,97 % 06.05.2020 Sbanken ASA Boliglån 50 % 2 % 06.05.2020 Fana Sparebank Nettlån Bolig 75 % 2,14 % 06.05.2020 Nordea Direct Boliglån inntil 50 % 2,15 % 13.05.2020 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,44 % 13.05.2020 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,57 % 13.05.2020 Fana Sparebank Nettlån Bolig 50 % 1,73 % 13.05.2020 Sandnes Sparebank Nedbetalingslån innenfor 70 % 1,92 % 13.05.2020 BN Bank ASA Boliglån Alltid Fornøyd 50 % 1,97 % 20.05.2020 Bulder Bank (Sparebanke

## Mean pooling

In [35]:
emb_path = Path("embeddings/nb_sbert/maxlen_parts_flat_mean_pooling.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    nynorsk_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)
    
search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
sent_alignment_results["mean_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

### Inspiser matches

In [36]:
print_matches(matches, nynorsk_texts_flat, bokmål_texts_flat, stop_printing_at=20)


Søketekst: 
        ,må du først ha sendt inn ein søknad om lån og stipendfor det studieåret du får barn. , men har fått barn innan sju månader etter at du har fullført ein grad,sender du inn denne søknaden saman med dokumentasjon på graden du har fullført. Du sender inn denne søknaden og dokumentasjonen etter at du ha
Match:
        ,må du først ha sendt inn en søknad om lån og stipendfor det studieåret du får barn. , men har fått barn innen sju måneder etter at du har fullført en grad,sender du inn denne søknaden sammen med dokumentasjon på graden du har fullført. Du sender inn søknaden og dokumentasjonen etter at du har fått 
Likhet:
        0.9942946434020996
        
Indekser:
            Nynorsk/søketekst:  2
            Bokmål/treff:       9
_______________________________________________________________

Søketekst: 
        - Eg er stolt av arbeidsplassen min!
Match:
        - Jeg er stolt av arbeidsplassen min!
Likhet:
        0.970596194267273
        
Indekser:
            

## Max pooling

In [37]:
emb_path = Path("embeddings/nb_sbert/maxlen_parts_flat_max_pooling.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    nynorsk_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

sent_alignment_results["max_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

### Inspiser matches

In [38]:
# print_matches(matches, nynorsk_texts_flat, bokmål_texts_flat, stop_printing_at=50)
print_matches(non_matches, nynorsk_texts_flat, bokmål_texts_flat, stop_printing_at=50)


Søketekst: 
        "Alltid i jobb" "Mista/sagt opp jobben" "Komme i jobb" "Aldri i jobb"
Match:
        hvis han/hun er midlertidig arbeidsufør på grunn av sykdom eller ulykke.
Likhet:
        0.592862606048584
        
Indekser:
            Nynorsk/søketekst:  0
            Bokmål/treff:       5745
_______________________________________________________________

Søketekst: 
        '
Match:
        .
Likhet:
        0.8640571236610413
        
Indekser:
            Nynorsk/søketekst:  1
            Bokmål/treff:       38
_______________________________________________________________

Søketekst: 
        - barn av låntakaren sin ektefelle som bur saman med låntakaren minst 40 prosent av tida
Match:
        Barna må bo hos deg minst 40 prosent av tiden for at du skal få mer stipend og lån.
Likhet:
        0.7128099203109741
        
Indekser:
            Nynorsk/søketekst:  5
            Bokmål/treff:       967
_______________________________________________________________

Søketeks

#### Falsk negativ

In [39]:
i = 90
print(search_result[i])
print(nynorsk_texts_flat[i])
print("\n___________\n")
print(bokmål_texts_flat[search_result[i][0]["corpus_id"]])

[{'corpus_id': 283, 'score': 0.9358025789260864}]
2 heimreiser

___________

2 hjemreiser


## Konklusjon
Vi får flest matches med naiv cutoff (vi vet at 71.07% av avsnittene er innenfor modellens makslengde)  
Vi har ikke gulldata å sammenlikne med, så det er ikke så godt å si helt sikkert om dette er den beste metoden.

In [41]:
pd.DataFrame(sent_alignment_results).T.sort_values("matches")

,matches,percent of sents,threshold
string_comparison,341.0,0.08,NaN
naiv_cutoff_biter,2237.0,60.61,0.95
max_pooling_biter,2252.0,61.01,0.95
mean_pooling_biter,2294.0,62.15,0.95
naiv_cutoff,2343.0,63.48,0.95
